
# 27A — V5 Sealed CONFIRM80 Protocol Freeze + Blind Event Worklist

Run this **only after 26A passed**.

The final evaluation protocol JSON must already be placed in the repository before this notebook runs.
This notebook then opens only the sealed **identity roster** and creates blind event-research worklists.

It does **not**:
- generate astrology,
- score the frozen candidate,
- score Control,
- inspect pairability,
- inspect 1–5 year gaps,
- change CONFIRM membership.


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json
import numpy as np
import pandas as pd

NOTEBOOK_VERSION="SAJU_ML_V5_CONFIRM_PROTOCOL_WORKLIST_20260817"
BATCH_COUNT=4
BATCH_SEED="V5_CONFIRM_EVENT_BATCH_20260817"

def repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repository.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

ROOT=repo_root()
FINAL_DIR=ROOT/"research/ml/artifacts/v5_final_roster"
CONFIRM_ROSTER=FINAL_DIR/"V5_CONFIRM_SUBJECT_ROSTER_80_SEALED.csv"
FINAL_FREEZE=FINAL_DIR/"V5_FINAL_ROSTER_FREEZE_DECISION.json"

CONTROL_DEC=ROOT/"research/ml/artifacts/v5_candidate_vs_control/V5_CANDIDATE_VS_PRODUCTION_CONTROL_DECISION.json"
CAND_SPEC=ROOT/"research/ml/artifacts/v5_tg10_multitask_balanced/V5_25A_FROZEN_CANDIDATE_MODEL_SPEC.json"
CAND_COEF=ROOT/"research/ml/artifacts/v5_tg10_multitask_balanced/V5_25A_FROZEN_CANDIDATE_COEFFICIENTS.csv"

ORIG_PROTOCOL=ROOT/"research/ml_corpus/v5_ground_truth/V5_LOCAL_MATCHED_EVENT_PROTOCOL.json"
CONFIRM_PROTOCOL=ROOT/"research/ml_corpus/v5_ground_truth/V5_CONFIRM_ONE_SHOT_EVALUATION_PROTOCOL.json"

OUT=ROOT/"research/ml/artifacts/v5_confirm_event_collection"
OUT.mkdir(parents=True,exist_ok=True)

for p in [CONFIRM_ROSTER,FINAL_FREEZE,CONTROL_DEC,CAND_SPEC,CAND_COEF,ORIG_PROTOCOL,CONFIRM_PROTOCOL]:
    if not p.exists(): raise FileNotFoundError(p)

control_dec=json.load(open(CONTROL_DEC,encoding="utf-8"))
freeze=json.load(open(FINAL_FREEZE,encoding="utf-8"))
orig=json.load(open(ORIG_PROTOCOL,encoding="utf-8"))
confirm_protocol=json.load(open(CONFIRM_PROTOCOL,encoding="utf-8"))
cand_spec=json.load(open(CAND_SPEC,encoding="utf-8"))

assert control_dec["status"]=="V5_FROZEN_CANDIDATE_BEATS_CONTROL_READY_FOR_CONFIRM_PROTOCOL"
assert control_dec["confirm_protocol_may_begin"] is True
assert cand_spec["status"]=="FROZEN_BEFORE_CONTROL_AND_CONFIRM"
assert cand_spec["architecture"]=="TG10_MULTITASK_EFFECT_RIDGE_BALANCED"
assert cand_spec["axis_required_at_inference"] is False
assert cand_spec["coefficients_sha256"]==sha256_file(CAND_COEF)

assert orig["version"]=="V5_LOCAL_MATCHED_EVENT_PROTOCOL_V1"
assert orig["rosters"]["CONFIRM"]=={"COMPETITIVE":20,"PROJECT":25,"STATUS":35,"TOTAL":80}
assert confirm_protocol["status"]=="PREDECLARED_AFTER_DEV_CANDIDATE_AND_CONTROL_GATE_BEFORE_CONFIRM_ROSTER_OPEN"

# The historic final-roster decision is authoritative for the sealed roster hash.
assert int(freeze["confirm_n"])==80
assert freeze["confirm_roster_sha256"]==sha256_file(CONFIRM_ROSTER)

print("26A gate:",control_dec["status"])
print("Candidate:",cand_spec["architecture"])
print("CONFIRM protocol SHA:",sha256_file(CONFIRM_PROTOCOL))
print("Opening sealed identity roster only.")


26A gate: V5_FROZEN_CANDIDATE_BEATS_CONTROL_READY_FOR_CONFIRM_PROTOCOL
Candidate: TG10_MULTITASK_EFFECT_RIDGE_BALANCED
CONFIRM protocol SHA: 55a3340c5cf9e52eeffec5708ded805997f5922ad515a1d5cbf86ca1141a0ee9
Opening sealed identity roster only.


## 1. Open and validate the sealed 80-person roster

In [2]:

confirm=pd.read_csv(CONFIRM_ROSTER)
assert len(confirm)==80
assert confirm.subject_id.nunique()==80
assert set(confirm.preassigned_axis)=={"COMPETITIVE","PROJECT","STATUS"}

counts=confirm.preassigned_axis.value_counts().to_dict()
expected={"COMPETITIVE":20,"PROJECT":25,"STATUS":35}
assert counts==expected,(counts,expected)

# No event/outcome columns should exist in the sealed identity roster.
for forbidden in ["event_year","polarity","event_type","astrology","control","pair"]:
    assert forbidden not in {c.lower() for c in confirm.columns}

print("CONFIRM counts:",counts)
print("Roster hash verified:",sha256_file(CONFIRM_ROSTER))


CONFIRM counts: {'STATUS': 35, 'PROJECT': 25, 'COMPETITIVE': 20}
Roster hash verified: 3144f52a8e9304c5cf1773a1dc346b58a1e41fa6af784b3a4ea64a6d318dbfac


## 2. Deterministic blind batch assignment

In [3]:

def det_hash(s):
    return hashlib.sha256((BATCH_SEED+"|"+str(s)).encode()).hexdigest()

parts=[]
for axis,g in confirm.groupby("preassigned_axis",sort=True):
    x=g.copy()
    x["_det"]=x.subject_id.astype(str).map(det_hash)
    x=x.sort_values(["_det","subject_id"]).reset_index(drop=True)
    # Round-robin within axis to mix each batch without using outcomes.
    x["batch_id"]=[(i%BATCH_COUNT)+1 for i in range(len(x))]
    parts.append(x)

work=pd.concat(parts,ignore_index=True)
# Deterministic within-batch order, still independent of outcomes.
work["_order_hash"]=work.subject_id.astype(str).map(lambda s:det_hash("ORDER|"+s))
work=work.sort_values(["batch_id","_order_hash"]).reset_index(drop=True)
work["research_order_in_batch"]=work.groupby("batch_id").cumcount()+1

safe_preferred=[
    "subject_id","name","preassigned_axis","candidate_source",
    "birth_date","birth_place","gender","wikidata_id","rodden_rating","source_row_key"
]
safe=[c for c in safe_preferred if c in work.columns]
outcols=["batch_id","research_order_in_batch"]+safe
worklist=work[outcols].copy()

assert len(worklist)==80
assert worklist.subject_id.nunique()==80

summary=(
    worklist.groupby(["batch_id","preassigned_axis"]).size()
    .rename("n").reset_index()
)
print(summary.pivot(index="batch_id",columns="preassigned_axis",values="n").fillna(0).astype(int))
print("batch sizes:",worklist.batch_id.value_counts().sort_index().to_dict())


preassigned_axis  COMPETITIVE  PROJECT  STATUS
batch_id                                      
1                           5        7       9
2                           5        6       9
3                           5        6       9
4                           5        6       8
batch sizes: {1: 21, 2: 20, 3: 20, 4: 19}


## 3. Write frozen worklists and blank event templates

In [4]:

master=OUT/"V5_CONFIRM_EVENT_WORKLIST_4BATCH.csv"
worklist.to_csv(master,index=False)

event_cols=confirm_protocol["event_collection"]["event_columns"]

for b in range(1,BATCH_COUNT+1):
    wb=worklist[worklist.batch_id==b].copy()
    wb.to_csv(OUT/f"V5_CONFIRM_BATCH_{b:02d}_SUBJECT_WORKLIST.csv",index=False)

    # Blank intake rows carry identity keys only; researcher adds event rows as needed.
    template=pd.DataFrame(columns=event_cols)
    template.to_csv(OUT/f"V5_CONFIRM_BATCH_{b:02d}_EVENTS_TEMPLATE.csv",index=False)

manifest={
    "version":"V5_CONFIRM_EVENT_WORKLIST_FREEZE_V1",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":"V5_CONFIRM_EVENT_WORKLIST_FROZEN_READY_FOR_BLIND_EVENT_COLLECTION",
    "confirm_n":80,
    "axis_counts":counts,
    "batch_count":BATCH_COUNT,
    "batch_sizes":worklist.batch_id.value_counts().sort_index().to_dict(),
    "confirm_roster_sha256":sha256_file(CONFIRM_ROSTER),
    "historic_final_freeze_sha256":sha256_file(FINAL_FREEZE),
    "26A_control_decision_sha256":sha256_file(CONTROL_DEC),
    "candidate_spec_sha256":sha256_file(CAND_SPEC),
    "candidate_coefficients_sha256":sha256_file(CAND_COEF),
    "original_event_protocol_sha256":sha256_file(ORIG_PROTOCOL),
    "confirm_evaluation_protocol_sha256":sha256_file(CONFIRM_PROTOCOL),
    "master_worklist_sha256":sha256_file(master),
    "rules":{
        "membership_changed":False,
        "subject_replacement_allowed":False,
        "astrology_visible_to_event_research":False,
        "Control_visible_to_event_research":False,
        "pairability_or_year_gap_visible_to_event_research":False,
        "chronology_balancing_used_for_research":False,
        "preassigned_axis_immutable":True,
        "record_all_eligible_major_events":True
    },
    "next_rule":"Research all four frozen batches under the blind guide. Freeze the complete CONFIRM event corpus before any pairing, astrology, candidate scoring, or Control scoring."
}
manifest_path=OUT/"V5_CONFIRM_EVENT_WORKLIST_FREEZE_DECISION.json"
json.dump(manifest,open(manifest_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps({
    "status":manifest["status"],
    "confirm_n":80,
    "axis_counts":counts,
    "batch_sizes":manifest["batch_sizes"]
},ensure_ascii=False,indent=2))


{
  "status": "V5_CONFIRM_EVENT_WORKLIST_FROZEN_READY_FOR_BLIND_EVENT_COLLECTION",
  "confirm_n": 80,
  "axis_counts": {
    "STATUS": 35,
    "PROJECT": 25,
    "COMPETITIVE": 20
  },
  "batch_sizes": {
    "1": 21,
    "2": 20,
    "3": 20,
    "4": 19
  }
}



## Next

Research **all 4 batches** using `V5_CONFIRM_EVENT_RESEARCH_GUIDE.md`.

Do not generate pairs or astrology yet.

For each batch, return:

```text
V5_CONFIRM_BATCH_NN_EVENTS.csv
V5_CONFIRM_BATCH_NN_SUBJECT_SWEEP_AUDIT.csv
V5_CONFIRM_BATCH_NN_RESEARCH_MANIFEST.json
```

After all four batches are complete, the next notebook will:
1. hard-QA and freeze the entire CONFIRM event corpus,
2. create PRIMARY/BROAD confidence sets,
3. collapse subject-axis-year-polarity,
4. create the frozen 1–5y pairs,
5. score the already-frozen V5 candidate and Production Control once,
6. issue PASS / INCONCLUSIVE / FAIL.

No tuning will occur after CONFIRM opens.
